# 03. 상권 밀집도 x 택시 빈차/수요 공간 분석

## 분석 배경 및 목적

택시의 공간적 수요 분포는 상권의 업종 구성 및 밀집도와 밀접한 관련이 있다. 음식점, 유흥업소가 밀집한 지역은 심야 택시 수요가 높고, 오피스 밀집 지역은 출퇴근 시간대에 수요가 집중된다. 한편 빈차(공차) 거리는 택시 운영 효율의 핵심 지표로, 상권 밀집도가 낮은 지역에서 빈차 거리가 길어지는 경향이 있다.

본 분석은 소상공인 상가업소 데이터(좌표, 업종)와 택시 하차 데이터를 500m 그리드 단위로 결합하여 다음을 검증한다.

1. **상권 밀집도와 빈차거리의 공간 상관**: 상가 밀집 지역일수록 빈차 거리가 짧은가?
2. **업종별 택시 수요 연관성**: 어떤 업종이 택시 수요와 가장 강한 상관을 보이는가?
3. **상권 유형별 시간대 패턴**: 오피스형, 유흥형, 주거형 상권의 택시 수요 시간 프로파일은 어떻게 다른가?

**방법론적 근거**: ACM (2016)의 연구는 택시 승하차 데이터의 공간 패턴을 활용하여 버스 노선을 최적화하였으며, 택시 하차 패턴이 해당 지역의 경제 활동 강도(상권 활성도)를 반영함을 실증하였다. 본 분석은 이 접근을 확장하여 상가업소의 업종 분포와 택시 수요의 공간 상관을 직접 분석한다.

- **내부**: DC_TBYXD012 (하차좌표 ALIGHT_POS_X/Y, 빈차거리 VACNTV_DIST)
- **외부**: 소상공인 상가업소 (별도 보유, `commercial/seoul_commercial.csv`)
- **주의**: 좌표계는 원본 로직(POS_Y/1e7)을 유지. 실제 반입 파일의 좌표 단위에 맞게 조정 필요.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import platform
import warnings
warnings.filterwarnings('ignore')

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)
from matplotlib.colors import Normalize
from scipy import stats

In [ ]:
import gc, psutil, os

def mem_usage(tag=''):
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'[MEM {tag}] {gb:.2f} GB')

CHUNK_SIZE = 1_000_000
mem_usage('start')

> **주의: POS_Y/1e7 변환은 원본 좌표 단위에 따라 다를 수 있음.**
> 첫 청크에서 좌표 범위(서울 위도 37.4~37.7, 경도 126.8~127.2)가 맞는지 반드시 확인하세요.

## 1. 상가업소 로드 + 그리드 밀집도

In [ ]:
D012_PATH = './DC_TBYXD012.csv'
COMMERCIAL = './external_data/commercial/seoul_commercial.csv'
GRID_LAT, GRID_LON = 0.0045, 0.0056   # 약 500m

shops = pd.read_csv(COMMERCIAL)
shops = shops.dropna(subset=['경도','위도'])
shops = shops[(shops['경도'].between(126, 128)) & (shops['위도'].between(37, 38))]
shops['grid_id'] = ((shops['위도']/GRID_LAT).astype(int).astype(str) + '_' +
                    (shops['경도']/GRID_LON).astype(int).astype(str))
print(f"상가업소 {len(shops):,}건, 업종대분류 {shops['업종대분류명'].nunique()}개")

grid_density = shops.groupby('grid_id').agg(
    shop_count=('상호명','count'), center_lat=('위도','mean'), center_lon=('경도','mean')).reset_index()
# 대표 업종
gcat = shops.groupby(['grid_id','업종대분류명']).size().reset_index(name='c')
dom = gcat.sort_values('c', ascending=False).drop_duplicates('grid_id')[['grid_id','업종대분류명']]
grid_density = grid_density.merge(dom.rename(columns={'업종대분류명':'dominant_category'}), on='grid_id', how='left')
grid_density['density_level'] = pd.qcut(grid_density['shop_count'], 5,
    labels=['매우낮음','낮음','보통','높음','매우높음'])
print(f"그리드 {len(grid_density):,}개, 평균 {grid_density['shop_count'].mean():.0f}개/그리드")

## 2. 청크 집계: 그리드별 하차건수·빈차거리 + 행정동별 승차

In [ ]:
usecols = ['RIDE_DTIME','PAY_AMT','VACNTV_DIST','ALIGHT_POS_X','ALIGHT_POS_Y','RIDE_A_CD']
dtypes  = {'RIDE_DTIME': str,'PAY_AMT':'float64','VACNTV_DIST':'float64',
           'ALIGHT_POS_X':'float64','ALIGHT_POS_Y':'float64','RIDE_A_CD': str}
grid_parts, gh_parts, dong_parts = [], [], []
total = 0
for chunk in pd.read_csv(D012_PATH, usecols=usecols, dtype=dtypes, chunksize=CHUNK_SIZE):
    chunk['alat'] = chunk['ALIGHT_POS_Y'] / 1e7
    chunk['alon'] = chunk['ALIGHT_POS_X']
    chunk = chunk[(chunk['alon'].between(126.7,127.2)) & (chunk['alat'].between(37.4,37.7))]
    if not len(chunk): 
        del chunk; gc.collect(); continue
    rd = pd.to_datetime(chunk['RIDE_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    chunk['hour'] = rd.dt.hour
    chunk['grid_id'] = ((chunk['alat']/GRID_LAT).astype(int).astype(str) + '_' +
                        (chunk['alon']/GRID_LON).astype(int).astype(str))
    total += len(chunk)
    grid_parts.append(chunk.groupby('grid_id').agg(
        alight_count=('PAY_AMT','size'), vac_sum=('VACNTV_DIST','sum'), fare_sum=('PAY_AMT','sum')).reset_index())
    gh_parts.append(chunk.groupby(['grid_id','hour']).size().reset_index(name='count'))
    dong_parts.append(chunk.groupby('RIDE_A_CD').agg(
        ride_count=('PAY_AMT','size'), fare_sum=('PAY_AMT','sum'), vac_sum=('VACNTV_DIST','sum')).reset_index())
    del chunk, rd; gc.collect()

taxi_grid = pd.concat(grid_parts).groupby('grid_id').sum().reset_index()
taxi_grid['avg_vacancy_dist'] = taxi_grid['vac_sum'] / taxi_grid['alight_count']
taxi_grid['avg_fare'] = taxi_grid['fare_sum'] / taxi_grid['alight_count']
grid_hour = pd.concat(gh_parts).groupby(['grid_id','hour'])['count'].sum().reset_index()
taxi_dong = pd.concat(dong_parts).groupby('RIDE_A_CD').sum().reset_index()
del grid_parts, gh_parts, dong_parts; gc.collect()
print(f"서울범위 {total:,}건, 택시 그리드 {len(taxi_grid):,}개"); mem_usage('after load')

## 3. 상권 밀집도 vs 빈차거리

그리드별 상가업소 수(밀집도)와 평균 빈차거리의 관계를 산점도와 상관계수로 분석한다. 음의 상관이 예상되며, 이는 "상권이 활성화된 지역일수록 다음 승객을 빨리 만난다"는 가설을 검증한다. ACM (2016)의 연구에서도 택시 수요 밀집 지역과 상업 활동 밀집 지역의 공간적 일치를 보고하였다.

In [ ]:
analysis = grid_density.merge(taxi_grid, on='grid_id', how='inner')
print(f"분석 그리드 {len(analysis):,}개 (상가+택시 공통)")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
sc = ax1.scatter(analysis['shop_count'], analysis['avg_vacancy_dist'], c=analysis['alight_count'],
                 cmap='YlOrRd', alpha=0.5, s=20,
                 norm=Normalize(vmin=0, vmax=analysis['alight_count'].quantile(0.95)))
plt.colorbar(sc, ax=ax1, label='하차건수')
z = np.polyfit(analysis['shop_count'], analysis['avg_vacancy_dist'], 1)
xs = np.linspace(analysis['shop_count'].min(), analysis['shop_count'].max(), 100)
ax1.plot(xs, np.poly1d(z)(xs), 'b--', lw=2, label=f'기울기 {z[0]:.2f}')
r, p = stats.pearsonr(analysis['shop_count'], analysis['avg_vacancy_dist'])
ax1.set_title(f'상가밀집도 vs 빈차거리 (r={r:.3f})', fontweight='bold')
ax1.set_xlabel('그리드 상가 수'); ax1.set_ylabel('평균 빈차거리(m)'); ax1.legend(); ax1.grid(alpha=0.3)
ds = analysis.groupby('density_level', observed=True)['avg_vacancy_dist'].mean()
ax2.bar(ds.index.astype(str), ds.values, color=['#E3F2FD','#90CAF9','#42A5F5','#1E88E5','#0D47A1'])
ax2.set_title('밀집도 구간별 평균 빈차거리', fontweight='bold'); ax2.set_xlabel('밀집도'); ax2.set_ylabel('m'); ax2.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 4. 업종별 택시 수요 상관

업종 대분류(음식, 소매, 생활서비스, 숙박, 유흥 등)별 그리드 내 업소 수와 택시 하차건수의 상관을 분석한다. 업종별 상관 강도의 차이는 택시 수요의 목적지 특성을 드러내며, 향후 수요 예측 모형에서 POI(Point of Interest) 피처 선택의 근거가 된다.

In [ ]:
gcp = shops.groupby(['grid_id','업종대분류명']).size().unstack(fill_value=0)
ca = gcp.merge(taxi_grid[['grid_id','alight_count','avg_vacancy_dist']], on='grid_id', how='inner')
rows = []
for cat in gcp.columns:
    r1, _ = stats.pearsonr(ca[cat], ca['alight_count'])
    r2, _ = stats.pearsonr(ca[cat], ca['avg_vacancy_dist'])
    rows.append({'업종': cat, '하차_상관': round(r1,3), '빈차_상관': round(r2,3)})
corr_df = pd.DataFrame(rows).sort_values('하차_상관', ascending=False)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
ax1.barh(corr_df['업종'], corr_df['하차_상관'], color=['#4CAF50' if v>0 else '#F44336' for v in corr_df['하차_상관']])
ax1.axvline(0, color='black', lw=0.5); ax1.set_title('업종 vs 하차건수 상관', fontweight='bold'); ax1.grid(alpha=0.3, axis='x')
cv = corr_df.sort_values('빈차_상관')
ax2.barh(cv['업종'], cv['빈차_상관'], color=['#4CAF50' if v<0 else '#F44336' for v in cv['빈차_상관']])
ax2.axvline(0, color='black', lw=0.5); ax2.set_title('업종 vs 빈차거리 상관', fontweight='bold'); ax2.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()
corr_df

## 5. 상권 유형별 시간대 패턴

In [ ]:
type_map = {'음식':'음식점','소매':'소매','생활서비스':'생활서비스','부동산':'오피스/부동산',
            '의료':'의료','숙박':'유흥/숙박','스포츠':'스포츠/여가','학문/교육':'교육'}
analysis['area_type'] = analysis['dominant_category'].map(type_map).fillna('기타')
gh = grid_hour.merge(analysis[['grid_id','area_type']], on='grid_id', how='inner')
th = gh.groupby(['area_type','hour'])['count'].sum().reset_index()
top_types = analysis['area_type'].value_counts().head(5).index.tolist()
fig, ax = plt.subplots(figsize=(14, 7))
for t in top_types:
    s = th[th['area_type']==t]; tot = s['count'].sum()
    ax.plot(s['hour'], s['count']/tot*100, 'o-', lw=2, label=t)
ax.set_title('상권 유형별 택시 하차 시간대 분포', fontweight='bold'); ax.set_xlabel('시간대'); ax.set_ylabel('비중(%)')
ax.set_xticks(range(24)); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. 택시 핫스팟 vs 상가 핫스팟

Kernel Density Estimation(KDE) 또는 상위 N% 그리드를 기준으로 택시 수요 핫스팟과 상가 밀집 핫스팟의 공간적 일치도를 비교한다. 두 핫스팟이 일치하지 않는 지역은 잠재 수요 불일치 구간으로, 택시 재배치 또는 상권 개발 정책의 대상이 될 수 있다.

In [ ]:
tt = analysis.nlargest(20, 'alight_count'); ts = analysis.nlargest(20, 'shop_count')
overlap = set(tt['grid_id']) & set(ts['grid_id'])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
s1 = ax1.scatter(tt['center_lon'], tt['center_lat'], c=tt['alight_count'], cmap='Reds', s=100, edgecolors='black', lw=0.5)
plt.colorbar(s1, ax=ax1, label='하차건수'); ax1.set_title('택시 하차 Top20 그리드', fontweight='bold'); ax1.grid(alpha=0.3)
s2 = ax2.scatter(ts['center_lon'], ts['center_lat'], c=ts['shop_count'], cmap='Greens', s=100, edgecolors='black', lw=0.5)
plt.colorbar(s2, ax=ax2, label='상가수'); ax2.set_title('상가 밀집 Top20 그리드', fontweight='bold'); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"택시Top20 ∩ 상가Top20: {len(overlap)}개 ({len(overlap)/20*100:.0f}%)")

## 7. 요약

본 분석의 핵심 발견과 실무적 시사점을 정리한다.

**실무 활용**: 상권-택시 공간 상관 결과는 (1) 택시 대기소 위치 선정, (2) 빈차 재배치 알고리즘의 목적지 가중치 설계, (3) 신규 상권 개발 시 교통 수요 예측에 활용할 수 있다. 특히 업종별 상관 강도는 도시계획에서 토지이용-교통 연계 모형의 입력 파라미터로 활용 가능하다.

In [ ]:
rv, pv = stats.pearsonr(analysis['shop_count'], analysis['avg_vacancy_dist'])
rd_, pd_ = stats.pearsonr(analysis['shop_count'], analysis['alight_count'])
low = analysis[analysis['density_level']=='매우낮음']['avg_vacancy_dist'].mean()
high = analysis[analysis['density_level']=='매우높음']['avg_vacancy_dist'].mean()
print('=== 상권 밀집도 × 택시 요약 ===')
print(f"분석 그리드 {len(analysis):,}개 (500m)")
print(f"상가수 vs 하차건수 r={rd_:.3f} / 상가수 vs 빈차거리 r={rv:.3f}")
print(f"밀집도 매우낮음 빈차 {low:,.0f}m → 매우높음 {high:,.0f}m ({(low-high)/low*100:.1f}% 감소)")
print(f"핫스팟 일치율 {len(overlap)/20*100:.0f}%")

---

## References

1. Chen, C., Zhang, D., Li, N., & Zhou, Z.-H. (2016). Bus Routes Design and Optimization via Taxi Data Analytics. *Proceedings of the ACM SIGKDD International Conference on Knowledge Discovery and Data Mining (KDD '16)*.
2. Zheng, Y., Capra, L., Wolfson, O., & Yang, H. (2014). Urban Computing: Concepts, Methodologies, and Applications. *ACM Transactions on Intelligent Systems and Technology*, 5(3), 1-55.
3. 소상공인시장진흥공단 (2024). 상가(상권)정보 개방 데이터. https://www.data.go.kr